# Customer Churn Prediction & Retention Risk Analysis

**Objective:** Analyze customer churn patterns and build a machine-learning model to identify customers at higher risk of churn.

**Dataset:** Telecom customer churn dataset with 7,043 customer records.

**Final candidate model:** Tuned Random Forest  
**Test ROC-AUC:** 0.8431  
**5-fold CV ROC-AUC:** 0.8490 ± 0.0129  
**Churn recall:** 76.7%  
**Churn F1:** 62.9%

> Feature importance indicates predictive association, not causation.

## 1. Business Problem

Customer churn is an important business problem for subscription-based companies. The goal is to understand churn patterns, identify high-risk segments, compare classification models, and select a model suitable for proactive churn identification.

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", None)

## 3. Load Data

> Change the CSV path below if your dataset has a different filename.

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

## 4. Data Cleaning

`TotalCharges` is converted to numeric. Blank values are handled as zero, consistent with the analysis where these records had zero tenure.

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].replace(" ", "0.0"),
    errors="coerce"
).fillna(0.0)

df.isna().sum()

In [ ]:
df["Churn"].value_counts()

## 5. Exploratory Data Analysis

### Contract vs Churn

In [ ]:
contract_churn = pd.crosstab(
    df["Contract"], df["Churn"], normalize="index"
) * 100
contract_churn

### Internet Service vs Churn

In [ ]:
internet_churn = pd.crosstab(
    df["InternetService"], df["Churn"], normalize="index"
) * 100
internet_churn

In [ ]:
df["InternetService"].value_counts()

### Monthly Charges by Internet Service

In [ ]:
df.groupby("InternetService")["MonthlyCharges"].agg(
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
)

### Internet Service + Contract

In [ ]:
internet_contract_churn = pd.crosstab(
    [df["InternetService"], df["Contract"]],
    df["Churn"], normalize="index"
) * 100
internet_contract_churn

### Selected categorical churn rates

In [ ]:
categorical_features_eda = [
    "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "PaperlessBilling", "PaymentMethod"
]

for col in categorical_features_eda:
    print(f"--- {col} ---")
    print(df.groupby(col)["Churn"].apply(lambda x: (x == "Yes").mean()))
    print()

## 6. Feature Preparation

In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

print(X.shape)
X.dtypes

## 7. Stratified Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

## 8. Preprocessing Pipeline

In [ ]:
numeric_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

categorical_features = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod"
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

## 9. Logistic Regression Baseline

In [ ]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_pipeline.fit(X_train, y_train)
y_pred = logistic_pipeline.predict(X_test)
y_pred_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
roc_auc = roc_auc_score((y_test == "Yes").astype(int), y_pred_proba)
print("ROC-AUC:", roc_auc)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

### Logistic Regression Coefficients

In [ ]:
feature_names = logistic_pipeline.named_steps["preprocessor"].get_feature_names_out()
coefficients = logistic_pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

display(coef_df.sort_values("Coefficient", ascending=False).head(15))
display(coef_df.sort_values("Coefficient", ascending=True).head(15))

## 10. Multicollinearity Check

In [ ]:
corr_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
corr_matrix = X[corr_features].corr()
corr_matrix

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

### Test removing TotalCharges

In [ ]:
X_no_total = X.drop(columns=["TotalCharges"])

X_train_nt, X_test_nt, y_train_nt, y_test_nt = train_test_split(
    X_no_total, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features_nt = ["SeniorCitizen", "tenure", "MonthlyCharges"]

preprocessor_nt = ColumnTransformer([
    ("num", StandardScaler(), numeric_features_nt),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

logistic_pipeline_nt = Pipeline([
    ("preprocessor", preprocessor_nt),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_pipeline_nt.fit(X_train_nt, y_train_nt)
y_pred_nt = logistic_pipeline_nt.predict(X_test_nt)
y_pred_proba_nt = logistic_pipeline_nt.predict_proba(X_test_nt)[:, 1]

print(classification_report(y_test_nt, y_pred_nt))
print("ROC-AUC without TotalCharges:",
      roc_auc_score((y_test_nt == "Yes").astype(int), y_pred_proba_nt))

## 11. Logistic Regression Threshold Analysis

In [ ]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]
results = []

for threshold in thresholds:
    pred = np.where(y_pred_proba >= threshold, "Yes", "No")
    results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred, pos_label="Yes"),
        "Recall": recall_score(y_test, pred, pos_label="Yes"),
        "F1": f1_score(y_test, pred, pos_label="Yes")
    })

threshold_results = pd.DataFrame(results)
threshold_results

## 12. Decision Tree Baseline

In [ ]:
decision_tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

decision_tree_pipeline.fit(X_train, y_train)
y_pred_dt = decision_tree_pipeline.predict(X_test)
y_pred_proba_dt = decision_tree_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_dt))
print("ROC-AUC:", roc_auc_score((y_test == "Yes").astype(int), y_pred_proba_dt))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_dt))

## 13. Tuned Decision Tree

In [ ]:
param_grid_dt = {
    "model__max_depth": [3, 5, 7, 10, None],
    "model__min_samples_split": [2, 10, 20],
    "model__min_samples_leaf": [1, 5, 10]
}

grid_dt = GridSearchCV(
    decision_tree_pipeline, param_grid_dt, cv=5,
    scoring="roc_auc", n_jobs=-1
)
grid_dt.fit(X_train, y_train)

print("Best parameters:", grid_dt.best_params_)
print("Best CV ROC-AUC:", grid_dt.best_score_)

In [ ]:
best_dt = grid_dt.best_estimator_
y_pred_dt_tuned = best_dt.predict(X_test)
y_pred_proba_dt_tuned = best_dt.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_dt_tuned))
print("Test ROC-AUC:",
      roc_auc_score((y_test == "Yes").astype(int), y_pred_proba_dt_tuned))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_dt_tuned))

## 14. Random Forest Baseline

In [ ]:
random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

random_forest_pipeline.fit(X_train, y_train)
y_pred_rf = random_forest_pipeline.predict(X_test)
y_pred_proba_rf = random_forest_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score((y_test == "Yes").astype(int), y_pred_proba_rf))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))

## 15. Tuned Random Forest

In [ ]:
param_grid_rf = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [5, 10, None],
    "model__min_samples_leaf": [1, 5, 10],
    "model__max_features": ["sqrt", "log2"],
    "model__class_weight": [None, "balanced"]
}

grid_rf = GridSearchCV(
    random_forest_pipeline, param_grid_rf, cv=5,
    scoring="roc_auc", n_jobs=-1
)
grid_rf.fit(X_train, y_train)

print("Best parameters:", grid_rf.best_params_)
print("Best CV ROC-AUC:", grid_rf.best_score_)

In [ ]:
best_rf = grid_rf.best_estimator_

y_pred_rf_tuned = best_rf.predict(X_test)
y_pred_proba_rf_tuned = best_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf_tuned))
print("Test ROC-AUC:",
      roc_auc_score((y_test == "Yes").astype(int), y_pred_proba_rf_tuned))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf_tuned))

## 16. Random Forest Threshold Analysis

In [ ]:
thresholds_rf = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]
results_rf = []

for threshold in thresholds_rf:
    pred = np.where(y_pred_proba_rf_tuned >= threshold, "Yes", "No")
    results_rf.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred, pos_label="Yes"),
        "Recall": recall_score(y_test, pred, pos_label="Yes"),
        "F1": f1_score(y_test, pred, pos_label="Yes")
    })

threshold_results_rf = pd.DataFrame(results_rf)
threshold_results_rf

## 17. Final Cross-Validation

In [ ]:
cv_scores_rf = cross_val_score(
    best_rf, X_train, y_train, cv=5,
    scoring="roc_auc", n_jobs=-1
)

print("Fold ROC-AUC scores:", cv_scores_rf)
print("Mean ROC-AUC:", cv_scores_rf.mean())
print("Std ROC-AUC:", cv_scores_rf.std())

## 18. Model Comparison

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression", "Decision Tree",
        "Tuned Decision Tree", "Random Forest", "Tuned Random Forest"
    ],
    "Accuracy": [0.81, 0.72, 0.80, 0.79, 0.76],
    "Churn Precision": [0.657, 0.48, 0.63, 0.63, 0.532],
    "Churn Recall": [0.559, 0.492, 0.567, 0.487, 0.767],
    "Churn F1": [0.604, 0.48, 0.60, 0.55, 0.629],
    "ROC-AUC": [0.8420, 0.6477, 0.8275, 0.8185, 0.8431]
})
model_comparison

## 19. Feature Importance

In [ ]:
feature_names_rf = best_rf.named_steps["preprocessor"].get_feature_names_out()
importances_rf = best_rf.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names_rf,
    "Importance": importances_rf
}).sort_values("Importance", ascending=False)

importance_df.head(20)

### Permutation Importance

In [ ]:
perm_importance = permutation_importance(
    best_rf, X_test, y_test, scoring="roc_auc",
    n_repeats=10, random_state=42, n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm_importance.importances_mean,
    "Importance_Std": perm_importance.importances_std
}).sort_values("Importance_Mean", ascending=False)

perm_df.head(15)

## 20. Final Findings & Business Recommendations

### Key predictive features
Permutation importance identified **Contract, tenure, InternetService, TotalCharges, OnlineSecurity, TechSupport, and PaymentMethod** as the strongest original features.

### Final model
The tuned Random Forest achieved:
- **Test ROC-AUC:** 0.8431
- **5-fold CV ROC-AUC:** 0.8490 ± 0.0129
- **Churn recall:** 76.7%
- **Churn F1:** 62.9%

### Recommendations
1. Prioritize month-to-month customers for retention efforts.
2. Focus on early-tenure customers, who showed substantially higher observed churn.
3. Investigate the fiber optic + month-to-month segment, which showed 54.61% observed churn.
4. Investigate customers without OnlineSecurity or TechSupport.
5. Investigate electronic-check customers while controlling for other customer characteristics.

### Limitation
These findings represent predictive associations, not causal relationships. Production performance may differ from the held-out test results.